**Finesse — Gamified Budgeting & Financial Health Scorer**

Anggota:
1. CFCC319D6Y0190 - Patrick Nicxon Hutabarat - Full-Stack Web Developer
2. CDCC319D6X0998 - Dame Theresia Rejeki Sidauruk - Data Science
3. CDCC319D6X1254 - Cikita Natasya Br Sembiring - Data Scientist
4. CACC319D6Y0343 - Rayza Indafri Yahya - AI Engineer
5. CACC319D6Y1720 - Samuel Gautama Manik - AI Engineer

## **Langkah 1: Persiapan Data dan Simulasi Grup A/B**
Karena dataset belum memiliki label eksperimen, simulasi pembagian pengguna dilakukan ke dalam Grup A (Control - Aplikasi Lama) dan Grup B (Treatment - Aplikasi Baru dengan Notifikasi Pengingat). Library `scipy.stats` dipanggil untuk kebutuhan uji statistik pada tahap selanjutnya.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load dataset
df = pd.read_csv('finesse_dataset.csv')

# Simulasi A/B Testing: Membagi user secara acak ke Grup A dan Grup B
np.random.seed(42) # Agar hasil acak tetap sama setiap kali di-run
users = df['user_id'].unique()
grup_a_users = np.random.choice(users, size=len(users)//2, replace=False)

# Buat kolom baru 'ab_group'
df['ab_group'] = df['user_id'].apply(lambda x: 'Group A (Control)' if x in grup_a_users else 'Group B (Treatment)')

print("Pembagian Grup A dan Grup B selesai!")
display(df[['user_id', 'ab_group', 'financial_health_score']].head())

Pembagian Grup A dan Grup B selesai!


,user_id,ab_group,financial_health_score
0,CUST00001,Group A (Control),88.91
1,CUST00001,Group A (Control),87.11
2,CUST00001,Group A (Control),70.26
3,CUST00001,Group A (Control),67.04
4,CUST00001,Group A (Control),97.21


## **Langkah 2: Menghitung Rata-rata Metrik (Performa Sementara)**
Sebelum dievaluasi melalui uji statistik, nilai rata-rata dari metrik yang menjadi tolak ukur perlu diperiksa terlebih dahulu. Pada tahapan ini, dilakukan perbandingan rata-rata Skor Kesehatan Finansial (financial_health_score) secara langsung antara kedua grup.

In [ ]:
# Menghitung rata-rata skor per grup
summary_stats = df.groupby('ab_group')['financial_health_score'].mean().reset_index()
summary_stats.columns = ['Grup', 'Rata-rata Financial Health Score']

display(summary_stats)

,Grup,Rata-rata Financial Health Score
0,Group A (Control),90.988849
1,Group B (Treatment),90.698685


## **Langkah 3: Uji Statistik (T-Test)**
Meskipun perhitungan angka rata-rata menunjukkan adanya selisih, perlu dibuktikan secara ilmiah apakah perbedaan tersebut signifikan secara statistik atau sekadar kebetulan. Uji hipotesis dilakukan menggunakan Independent T-Test. Jika nilai P-Value berada di bawah 0.05, dapat disimpulkan bahwa fitur baru di Grup B memberikan dampak yang nyata dan terukur.

In [ ]:
# Memisahkan data skor antara Grup A dan Grup B
skor_grup_a = df[df['ab_group'] == 'Group A (Control)']['financial_health_score']
skor_grup_b = df[df['ab_group'] == 'Group B (Treatment)']['financial_health_score']

# Melakukan T-Test
t_stat, p_value = stats.ttest_ind(skor_grup_b, skor_grup_a)

print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4f}")
print("-" * 30)

# Menarik kesimpulan berdasarkan P-Value
alpha = 0.05
if p_value < alpha:
    if t_stat > 0:
        print("KESIMPULAN: Fitur baru BERHASIL! Grup B memiliki skor signifikan lebih TINGGI.")
    else:
        print("KESIMPULAN: Fitur baru BERDAMPAK NEGATIF! Grup B memiliki skor signifikan lebih RENDAH.")
else:
    print("KESIMPULAN: Tidak ada perbedaan signifikan. Perbedaan skor mungkin hanya KEBETULAN semata.")

T-Statistic: -1.1018
P-Value: 0.2706
------------------------------
KESIMPULAN: Tidak ada perbedaan signifikan. Perbedaan skor mungkin hanya KEBETULAN semata.
